# PANDA training — thin Kaggle wrapper

Clones the repo, installs deps, runs `python -m src.train` on the chosen fold. All the actual logic lives in `src/`.

**Inputs (Kaggle datasets to attach):**
1. `panda-resized-train-data-512x512` (xhlulu)
2. (No competition data needed if using xhlulu's preprocessed thumbnails)

**Settings:** GPU T4 ON, Internet ON (need it to git clone + pip install + download EfficientNet weights).

In [ ]:
# Replace with your team's repo URL and branch
REPO   = 'https://github.com/<your-team>/panda.git'
BRANCH = 'main'
FOLD   = 0    # change for other folds
EPOCHS = 6

In [ ]:
import subprocess, os
if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
!pip install -q efficientnet_pytorch

In [ ]:
import glob, os

# Auto-detect the xhlulu image dir (path can vary between Kaggle layouts)
candidates = (glob.glob('/kaggle/input/*resized*512*/train_images/train_images') +
              glob.glob('/kaggle/input/*resized*512*/train_images') +
              glob.glob('/kaggle/input/**/train_images', recursive=True))
IMAGE_DIR = next((c for c in candidates if os.path.isdir(c) and len(os.listdir(c)) > 5000), None)
if IMAGE_DIR is None:
    raise RuntimeError(f'Could not find image dir. /kaggle/input contains: {os.listdir("/kaggle/input")}')
print('IMAGE_DIR:', IMAGE_DIR, 'files:', len(os.listdir(IMAGE_DIR)))

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/repo')

import subprocess
cmd = [
    'python', '-m', 'src.train',
    '--fold', str(FOLD),
    '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
    '--image-dir', IMAGE_DIR,
    '--epochs', str(EPOCHS),
    '--output-dir', '/kaggle/working',
]
subprocess.run(cmd, cwd='/kaggle/working/repo', check=True)

In [ ]:
# List output weights
import os
for f in os.listdir('/kaggle/working'):
    if f.endswith('.pth'):
        print(f, os.path.getsize(f'/kaggle/working/{f}') / 1e6, 'MB')